# Quick Start -- Langgraph

## 核心抽象：**图**
* 将每个应用抽象成 一个graph
* 图里有 节点（node）（可以是调用 LLM、执行函数、调用 API 等）
* 节点之间通过边（edge）连接，边可以是有向的也可以是无向的
* 整个执行过程由 状态（state） 驱动

## 核心组件

1. 状态 state
* 是一个 Python dict 或者自定义数据结构
* 用来保存对话上下文，比如用户问题、LLM 回复、中间变量

2. 节点 node
* 图中的一个步骤：可以是调用 LLM、执行函数、调用 API 等


3. 图 graph
* 把多个 node 连起来形成一个流程
```python
from langgraph.graph import StateGraph, END

builder = StateGraph(State)
builder.add_node("llm", call_llm)
builder.set_entry_point("llm")   # 起点
builder.add_edge("llm", END)     # 结束
graph = builder.compile()
```


## 学习 graph

* StateGraph
    * 最常用，适合 有状态 的应用（比如 Agent、对话机器人）
    * 通过state判断步骤

* MessageGraph
    * 是 StateGraph 的一个简化版本，专门为 对话消息流 设计
    * 状态自动就是一个消息列表
    
* CompiledGraph
    * 编译后的图，不能再添加节点或边
    * 可以直接运行

* SubGraph
    * 就是图的嵌套 形成子流程

In [ ]:
from langgraph.graph import StateGraph, END, START


def my_node(state):
    """
    节点函数
    """
    return {"x": state["x"] + 1, "y": state['y'] + 2}


builder =  StateGraph(dict)
builder.add_node(my_node)
builder.add_edge(START, "my_node")  # 添加一条边 从 START 到 my_node

# 编译图
graph = builder.compile()
print(graph)

# 运行图
state = {"x": 1, "y": 2}
res = graph.invoke(state)
print(res, state)   # 可以发现 state 没有改变 {'x': 2, 'y': 4} {'x': 1, 'y': 2}


{'x': 2, 'y': 4} {'x': 1, 'y': 2}


## state 状态
〉LangGraph 强烈推荐用 TypedDict 或 Pydantic 模型来定义 State